# BirdCLEF 2026 - Perch Only Inference
Offline-friendly native Perch classifier notebook using the Kaggle-compatible model export.

In [ ]:
import os
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')
PERCH_MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

SAMPLE_RATE = 32000
WINDOW_SEC = 5.0
BATCH_SIZE = 16

print('TF version:', tf.__version__)
print('Perch model dir:', PERCH_MODEL_DIR)

In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES = taxonomy_df['primary_label'].astype(str).tolist()
ZERO_SHOT_PRIOR = 1.0 / len(SPECIES)

label_file = PERCH_MODEL_DIR / 'assets' / 'labels.csv'
if not label_file.exists():
    assets_dir = PERCH_MODEL_DIR / 'assets'
    csvs = sorted(assets_dir.glob('*.csv')) if assets_dir.exists() else []
    assert len(csvs) > 0, f'No label CSV found in {assets_dir}'
    label_file = csvs[0]

label_df = pd.read_csv(label_file)
ebird_col = label_df.columns[0]
for col in ['ebird2021', 'ebird_code', 'species_code']:
    if col in label_df.columns:
        ebird_col = col
        break

perch_labels = label_df[ebird_col].astype(str).tolist()
perch_to_idx = {species: idx for idx, species in enumerate(perch_labels)}
competition_index = np.array([perch_to_idx.get(species, -1) for species in SPECIES], dtype=np.int32)
mapped_mask = competition_index >= 0

print(f'Perch classes: {len(perch_labels)}')
print(f'Mapped competition species: {int(mapped_mask.sum())}/{len(SPECIES)}')

In [ ]:
perch = tf.saved_model.load(str(PERCH_MODEL_DIR))
infer_fn = perch.signatures['serving_default']

test_wav = tf.constant(np.zeros((1, int(SAMPLE_RATE * WINDOW_SEC)), dtype=np.float32))
test_out = infer_fn(inputs=test_wav)
assert 'label' in test_out, f"Expected 'label' key, got: {list(test_out.keys())}"
print(f"Perch loaded ✅ | classes: {test_out['label'].shape[-1]} | keys: {list(test_out.keys())}")

In [ ]:
def parse_row_id(row_id):
    stem, end_time = row_id.rsplit('_', 1)
    return stem, int(end_time)

def find_audio_file(soundscape_dir, file_stem):
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            return candidate
    return None

soundscape_dir = DATA_DIR / 'test_soundscapes'
window_samples = int(SAMPLE_RATE * WINDOW_SEC)

file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

row_frames = []
t_start = time.time()

for file_idx, file_stem in enumerate(sorted(file_windows.keys()), start=1):
    audio_path = find_audio_file(soundscape_dir, file_stem)
    if audio_path is None:
        row_ids = [row_id for row_id, _ in file_windows[file_stem]]
        prior_block = np.full((len(row_ids), len(SPECIES)), ZERO_SHOT_PRIOR, dtype=np.float32)
        frame = pd.DataFrame(prior_block, columns=SPECIES)
        frame.insert(0, 'row_id', row_ids)
        row_frames.append(frame)
        continue

    audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    chunks = []
    row_ids = []
    for row_id, end_time in file_windows[file_stem]:
        start_sample = max(0, int((end_time - WINDOW_SEC) * SAMPLE_RATE))
        end_sample = int(end_time * SAMPLE_RATE)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        elif len(chunk) > window_samples:
            chunk = chunk[:window_samples]
        chunks.append(chunk.astype(np.float32, copy=False))
        row_ids.append(row_id)

    projected_batches = []
    for start_idx in range(0, len(chunks), BATCH_SIZE):
        batch_waveforms = np.stack(chunks[start_idx:start_idx + BATCH_SIZE], axis=0)
        output = infer_fn(inputs=tf.constant(batch_waveforms, dtype=tf.float32))
        logits = output['label'].numpy()
        probs = 1.0 / (1.0 + np.exp(-logits))

        projected = np.full((probs.shape[0], len(SPECIES)), ZERO_SHOT_PRIOR, dtype=np.float32)
        projected[:, mapped_mask] = probs[:, competition_index[mapped_mask]]
        projected_batches.append(projected)

    frame = pd.DataFrame(np.vstack(projected_batches), columns=SPECIES)
    frame.insert(0, 'row_id', row_ids)
    row_frames.append(frame)

    if file_idx % 10 == 0:
        elapsed = time.time() - t_start
        print(f'{file_idx}/{len(file_windows)} soundscapes | {elapsed/60:.1f} min elapsed')

submission = pd.concat(row_frames, ignore_index=True)
submission = submission.set_index('row_id').reindex(sample_sub['row_id']).reset_index()
submission.columns = sample_sub.columns
submission = submission.fillna(ZERO_SHOT_PRIOR)

assert set(submission['row_id']) == set(sample_sub['row_id'])
submission.to_csv(OUTPUT_PATH, index=False)

elapsed = time.time() - t_start
print(f'Submission saved: {OUTPUT_PATH}')
print(f'Shape: {submission.shape}')
print(f'Total time: {elapsed/60:.1f} min')
submission.head()